# 04 SHAP Analysis

Google Colab notebook version.

In [ ]:
# Install required packages
!pip -q install shap openpyxl

In [ ]:

# ============================================================
# SHAP ANALYSIS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

def summarize_time(X):
    return X.mean(axis=1)

def run_kernel_shap_feature_importance(
    model,
    X_train,
    X_test,
    features,
    window=48,
    background_size=128,
    explain_size=256,
    nsamples=100,

):
    X_bg = summarize_time(X_train[:background_size])
    X_explain = summarize_time(X_test[:explain_size])

    def model_predict(x):
        x_seq = np.tile(x[:, None, :], (1, window, 1))
        return model.predict(x_seq, verbose=0).flatten()

    explainer = shap.KernelExplainer(model_predict, X_bg)
    shap_values = explainer.shap_values(X_explain, nsamples=nsamples)

    shap_values_array = np.asarray(shap_values)
    mean_abs = np.mean(np.abs(shap_values_array), axis=0)

    importance_df = pd.DataFrame({
        "Feature": features,
        "MeanAbsSHAP": mean_abs,
    }).sort_values("MeanAbsSHAP", ascending=False)

    importance_df.to_csv(output_csv, index=False)

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_explain, feature_names=features, show=False)
    plt.tight_layout()
    plt.savefig(output_fig, dpi=500, bbox_inches="tight")
    plt.show()

    print(f"Saved SHAP feature importance to: {output_csv}")
    print(f"Saved SHAP summary figure to: {output_fig}")

    return shap_values, importance_df

def run_temporal_shap_profile(
    model,
    X_train,
    X_test,
    features,
    background_size=100,
    explain_size=300,

):
    explainer = shap.GradientExplainer(model, X_train[:background_size])
    shap_values = explainer.shap_values(X_test[:explain_size])

    if isinstance(shap_values, list):
        shap_values = shap_values[0]

    shap_values = np.asarray(shap_values)

    temporal_profile = np.mean(np.abs(shap_values), axis=(0, 2))
    temporal_df = pd.DataFrame({
        "Lag_position": np.arange(len(temporal_profile)),
        "MeanAbsSHAP": temporal_profile,
    })
    temporal_df.to_csv(output_temporal_csv, index=False)

    feature_time_matrix = np.mean(np.abs(shap_values), axis=0)
    matrix_df = pd.DataFrame(feature_time_matrix, columns=features)
    matrix_df.to_csv(output_matrix_csv, index=False)



    return temporal_df, matrix_df

def plot_temporal_shap_profile(

):
    df = pd.read_csv(temporal_csv)

    plt.figure(figsize=(7.2, 4.6))
    plt.plot(df["Lag_position"], df["MeanAbsSHAP"], linewidth=1.6, marker="o", markersize=4)
    plt.xlabel("Lag position in 48-hour input window")
    plt.ylabel("Mean |SHAP|")
    plt.title("Temporal SHAP profile across the 48-hour input window")
    plt.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)
    plt.tight_layout()
    plt.savefig(output_fig, dpi=500, bbox_inches="tight")
    plt.show()

    print(f"Saved temporal SHAP figure to: {output_fig}")

def plot_feature_time_matrix(

):
    df = pd.read_csv(matrix_csv)

    plt.figure(figsize=(8.2, 5.8))
    plt.imshow(df.values.T, aspect="auto", origin="upper", interpolation="nearest")
    plt.xlabel("Lag position in 48-hour input window")
    plt.ylabel("Feature")
    plt.yticks(np.arange(len(df.columns)), df.columns)
    cbar = plt.colorbar()
    cbar.set_label("Mean |SHAP|")
    plt.title("Feature-time SHAP attribution matrix")
    plt.tight_layout()
    plt.savefig(output_fig, dpi=500, bbox_inches="tight")
    plt.show()
